In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change Questionnnair -> Questionnaire if needed.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "DIQ_I.xpt"
csv_file = DATA_DIR / "DIQ_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 DIQ_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "DIQ010",
    "DID040",
    "DIQ160",
    "DIQ170",
    "DIQ172",
    "DIQ175A",
    "DIQ175B",
    "DIQ175C",
    "DIQ175D",
    "DIQ175E",
    "DIQ175F",
    "DIQ175G",
    "DIQ175H",
    "DIQ175I",
    "DIQ175J",
    "DIQ175K",
    "DIQ175L",
    "DIQ175M",
    "DIQ175N",
    "DIQ175O",
    "DIQ175P",
    "DIQ175Q",
    "DIQ175R",
    "DIQ175S",
    "DIQ175T",
    "DIQ175U",
    "DIQ175V",
    "DIQ175W",
    "DIQ175X",
    "DIQ180",
    "DIQ050",
    "DID060",
    "DIQ060U",
    "DIQ070",
    "DIQ230",
    "DIQ240",
    "DID250",
    "DID260",
    "DIQ260U",
    "DIQ275",
    "DIQ280",
    "DIQ291",
    "DIQ300S",
    "DIQ300D",
    "DID310S",
    "DID310D",
    "DID320",
    "DID330",
    "DID341",
    "DID350",
    "DIQ350U",
    "DIQ360",
    "DIQ080"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "DIQ_I columns/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 9575:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 54:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,575 rows x 54 columns."
)


# ============================================================
# 4. DEFINE DECIMAL / INTEGER FIELDS
# ============================================================

#
# IMPORTANT:
#
# DIQ_I is NOT an all-integer file.
#
# DIQ280:
#
#     "What was your last A1C level?"
#
# is a genuine continuous decimal variable.
#
#
# Examples:
#
#     5.7
#     6.3
#     7.5
#     18.5
#
#
# Therefore:
#
#     DIQ280 MUST remain float.
#
#
# All other 53 fields are IDs, codes, quantities,
# ages, BP values, LDL values, etc. released as
# whole-number values.
#

DECIMAL_COLUMNS = [
    "DIQ280"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(DECIMAL_COLUMNS) != 1:

    raise ValueError(
        "Expected exactly 1 decimal field."
    )


if len(INTEGER_COLUMNS) != 53:

    raise ValueError(
        "Expected exactly 53 integer/code fields."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT/pandas zero artifact:
#
#     5.397605346934028e-79
#
#
# NEVER blanket-replace it before confirming the
# exact variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 54 DIQ_I variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny values:",
    tiny_variable_count
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached DIQ_I.xpt:
#
#     DID250      13
#     DID260     188
#     DID341     242
#     DID350     139
#
# TOTAL         582
#
#
# CDC:
#
# DID250:
#     0 = None
#
# DID260:
#     0 = Never
#
# DID341:
#     0 = None
#
# DID350:
#     0 = None
#

EXPECTED_TINY_COUNTS = {
    "DID250": 13,
    "DID260": 188,
    "DID341": 242,
    "DID350": 139
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated DIQ_I attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 582:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if tiny_variable_count != 4:

    raise ValueError(
        "Expected tiny values in exactly 4 variables."
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed before repair."
    )


print(
    "\nPASS: Exact 582-value tiny-zero pattern "
    "across 4 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 582:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED XPT COPY
# ============================================================

#
# expected_df becomes the exact numeric fidelity reference.
#
# ONLY:
#
#     validated tiny -> 0
#
# is allowed to differ from source_df.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is used ONLY for detection.
#
# It does NOT modify data.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


detected_decimal_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs.\n\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Actual:\n{detected_decimal_columns}"
    )


print(
    "\nPASS: DIQ280 is the ONLY genuine "
    "decimal variable."
)


# ============================================================
# 11. DIQ280 DECIMAL DETAILS
# ============================================================

a1c = df["DIQ280"]


a1c_nonmissing = int(
    a1c.notna().sum()
)


a1c_missing = int(
    a1c.isna().sum()
)


a1c_values = (
    a1c.dropna()
    .to_numpy(
        dtype=float
    )
)


a1c_fractional = int(
    (
        ~np.isclose(
            a1c_values,
            np.round(a1c_values),
            rtol=0,
            atol=1e-12
        )
    )
    .sum()
)


a1c_whole_looking = (
    a1c_nonmissing
    -
    a1c_fractional
)


print(
    "\n--- DIQ280 DECIMAL DETAILS ---"
)

print(
    "Nonmissing:",
    f"{a1c_nonmissing:,}"
)

print(
    "Missing:",
    f"{a1c_missing:,}"
)

print(
    "Fractional observations:",
    f"{a1c_fractional:,}"
)

print(
    "Whole-looking observations:",
    f"{a1c_whole_looking:,}"
)


if a1c_nonmissing != 641:

    raise ValueError(
        "Unexpected DIQ280 nonmissing count."
    )


if a1c_missing != 8934:

    raise ValueError(
        "Unexpected DIQ280 missing count."
    )


if a1c_fractional != 269:

    raise ValueError(
        "Unexpected DIQ280 fractional count."
    )


if a1c_whole_looking != 372:

    raise ValueError(
        "Unexpected DIQ280 whole-looking count."
    )


print(
    "PASS: DIQ280 decimal structure validated."
)


# ============================================================
# 12. CDC CHECK - DIQ280
# ============================================================

#
# CDC:
#
#     2 to 18.5 = 404
#     777       = 2   Refused
#     999       = 235 Don't know
#
#     Missing   = 8,934
#

a1c_valid_range = int(
    df["DIQ280"]
    .between(
        2,
        18.5
    )
    .sum()
)


a1c_777 = int(
    df["DIQ280"]
    .eq(777)
    .sum()
)


a1c_999 = int(
    df["DIQ280"]
    .eq(999)
    .sum()
)


print(
    "\n--- DIQ280 CDC CHECK ---"
)

print(
    "2-18.5:",
    a1c_valid_range
)

print(
    "777:",
    a1c_777
)

print(
    "999:",
    a1c_999
)

print(
    "Missing:",
    a1c_missing
)


if a1c_valid_range != 404:

    raise ValueError(
        "DIQ280 valid A1C range count differs."
    )


if a1c_777 != 2:

    raise ValueError(
        "DIQ280 refused count differs."
    )


if a1c_999 != 235:

    raise ValueError(
        "DIQ280 don't-know count differs."
    )


if a1c_missing != 8934:

    raise ValueError(
        "DIQ280 missing count differs."
    )


print(
    "PASS: DIQ280 matches CDC."
)


# ============================================================
# 13. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"].isna().sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"].duplicated().sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(df["SEQN"].isna().sum()) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(df["SEQN"].duplicated().sum()) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 14. VALIDATE 53 WHOLE-NUMBER FIELDS
# ============================================================

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    if not np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    ).all():

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: All 53 ID/code/count fields "
    "are whole-number."
)


# ============================================================
# 15. CONVERT PROVEN INTEGER FIELDS TO Int64
# ============================================================

#
# DIQ280 is intentionally excluded.
#
# NO round() during conversion.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: 53 proven whole-number fields "
    "converted to nullable Int64."
)


# ============================================================
# 16. CONFIRM DIQ280 REMAINS FLOAT
# ============================================================

if not pd.api.types.is_float_dtype(
    df["DIQ280"].dtype
):

    raise ValueError(
        "DIQ280 is no longer floating-point."
    )


print(
    "PASS: DIQ280 remains floating-point."
)


# ============================================================
# 17. HELPER - FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequency differs.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs."
        )


    print(
        f"PASS: {column} matches expected frequencies."
    )


# ============================================================
# 18. CDC CHECK - DIQ010
# ============================================================

check_frequency(
    "DIQ010",
    {
        1: 856,
        2: 8568,
        3: 147,
        9: 4
    },
    0
)


# ============================================================
# 19. CDC CHECK - DID040
# ============================================================

#
# Age first told had diabetes:
#
#     2-78 = 833
#     80   = 7
#     666  = 1
#     999  = 12
#
# Missing = 8,722
#

did040_range = int(
    df["DID040"]
    .between(
        2,
        78
    )
    .sum()
)


if did040_range != 833:

    raise ValueError(
        "DID040 2-78 range count differs."
    )


if int(df["DID040"].eq(80).sum()) != 7:

    raise ValueError(
        "DID040 age-80 count differs."
    )


if int(df["DID040"].eq(666).sum()) != 1:

    raise ValueError(
        "DID040 <1 year count differs."
    )


if int(df["DID040"].eq(999).sum()) != 12:

    raise ValueError(
        "DID040 don't-know count differs."
    )


if int(df["DID040"].isna().sum()) != 8722:

    raise ValueError(
        "DID040 missing count differs."
    )


print(
    "PASS: DID040 matches CDC."
)


# ============================================================
# 20. SELECTED CORE DIABETES FREQUENCIES
# ============================================================

check_frequency(
    "DIQ160",
    {
        1: 513,
        2: 5521,
        9: 11
    },
    3530
)


check_frequency(
    "DIQ170",
    {
        1: 899,
        2: 5268,
        9: 19
    },
    3389
)


check_frequency(
    "DIQ172",
    {
        1: 1588,
        2: 4510,
        7: 1,
        9: 87
    },
    3389
)


check_frequency(
    "DIQ180",
    {
        1: 2836,
        2: 3167,
        7: 2,
        9: 181
    },
    3389
)


# ============================================================
# 21. DIQ175A-X RISK-FACTOR CODE CHECK
# ============================================================

#
# These are multi-response variables.
#
# A populated variable should contain ONLY its
# designated category code.
#

RISK_REASON_COUNTS = {
    "DIQ175A": (10, 1186),
    "DIQ175B": (11, 601),
    "DIQ175C": (12, 160),
    "DIQ175D": (13, 459),
    "DIQ175E": (14, 155),
    "DIQ175F": (15, 53),
    "DIQ175G": (16, 280),
    "DIQ175H": (17, 258),
    "DIQ175I": (18, 77),
    "DIQ175J": (19, 150),
    "DIQ175K": (20, 31),
    "DIQ175L": (21, 59),
    "DIQ175M": (22, 138),
    "DIQ175N": (23, 109),
    "DIQ175O": (24, 134),
    "DIQ175P": (25, 117),
    "DIQ175Q": (26, 136),
    "DIQ175R": (27, 23),
    "DIQ175S": (28, 31),
    "DIQ175T": (29, 105),
    "DIQ175U": (30, 79),
    "DIQ175V": (31, 11),
    "DIQ175W": (32, 4),
    "DIQ175X": (33, 1)
}


print(
    "\n--- DIQ175A-X RISK FACTOR CHECK ---"
)


for col, (expected_code, expected_count) in RISK_REASON_COUNTS.items():

    nonmissing = df[col].dropna()

    actual_count = int(
        nonmissing.shape[0]
    )


    observed_codes = set(
        nonmissing
        .astype(int)
        .unique()
        .tolist()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} "
            f"nonmissing, found {actual_count:,}."
        )


    if observed_codes != {expected_code}:

        raise ValueError(
            f"{col}: unexpected codes {observed_codes}."
        )


print(
    "PASS: DIQ175A-X response-category "
    "structure preserved."
)


# ============================================================
# 22. INSULIN / DIABETIC PILL CHECKS
# ============================================================

check_frequency(
    "DIQ050",
    {
        1: 256,
        2: 9316,
        7: 1,
        9: 2
    },
    0
)


check_frequency(
    "DIQ070",
    {
        1: 643,
        2: 870,
        7: 1,
        9: 1
    },
    8060
)


# ============================================================
# 23. DID250 ZERO + RANGE CHECK
# ============================================================

#
# CDC:
#
#     1-60 = 627
#     0    = 13 None
#     9999 = 3 Don't know
#
# Missing = 8,932
#

if int(
    df["DID250"]
    .between(1, 60)
    .sum()
) != 627:

    raise ValueError(
        "DID250 1-60 count differs."
    )


if int(
    df["DID250"]
    .eq(0)
    .sum()
) != 13:

    raise ValueError(
        "DID250 zero count differs."
    )


if int(
    df["DID250"]
    .eq(9999)
    .sum()
) != 3:

    raise ValueError(
        "DID250 9999 count differs."
    )


if int(
    df["DID250"]
    .isna()
    .sum()
) != 8932:

    raise ValueError(
        "DID250 missing count differs."
    )


print(
    "PASS: DID250 zero/range pattern validated."
)


# ============================================================
# 24. DIQ240 -> DID250 ROUTING
# ============================================================

did250_expected = (
    df["DIQ240"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


did250_present = (
    df["DID250"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        did250_expected
        !=
        did250_present
    )
) != 0:

    raise ValueError(
        "DID250 routing differs from DIQ240=1."
    )


print(
    "PASS: DIQ240/DID250 routing validated."
)


# ============================================================
# 25. DID260 ZERO + RANGE CHECK
# ============================================================

#
# CDC:
#
#     1-15 = 661
#     0    = 188 Never
#
# Missing = 8,726
#

if int(
    df["DID260"]
    .between(
        1,
        15
    )
    .sum()
) != 661:

    raise ValueError(
        "DID260 1-15 range count differs."
    )


if int(
    df["DID260"]
    .eq(0)
    .sum()
) != 188:

    raise ValueError(
        "DID260 zero count differs."
    )


if int(
    df["DID260"]
    .isna()
    .sum()
) != 8726:

    raise ValueError(
        "DID260 missing count differs."
    )


print(
    "PASS: DID260 zero/range pattern validated."
)


# ============================================================
# 26. DID260 / DIQ260U ROUTING CHECK
# ============================================================

#
# Unit should exist only when DID260 is positive.
#

did260_positive = (
    df["DID260"]
    .between(
        1,
        15
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


diq260u_present = (
    df["DIQ260U"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        did260_positive
        !=
        diq260u_present
    )
) != 0:

    raise ValueError(
        "DIQ260U routing differs from DID260."
    )


check_frequency(
    "DIQ260U",
    {
        1: 412,
        2: 150,
        3: 65,
        4: 34
    },
    8914
)


# ============================================================
# 27. DIQ275 / DIQ280 / DIQ291 ROUTING
# ============================================================

check_frequency(
    "DIQ275",
    {
        1: 641,
        2: 156,
        9: 56
    },
    8722
)


a1c_followup_expected = (
    df["DIQ275"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


diq280_present = (
    df["DIQ280"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


diq291_present = (
    df["DIQ291"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        a1c_followup_expected
        !=
        diq280_present
    )
) != 0:

    raise ValueError(
        "DIQ280 availability differs from DIQ275=1."
    )


if int(
    np.sum(
        a1c_followup_expected
        !=
        diq291_present
    )
) != 0:

    raise ValueError(
        "DIQ291 availability differs from DIQ275=1."
    )


print(
    "PASS: DIQ275/DIQ280/DIQ291 routing validated."
)


check_frequency(
    "DIQ291",
    {
        1: 213,
        2: 182,
        3: 32,
        4: 5,
        5: 9,
        6: 88,
        77: 2,
        99: 110
    },
    8934
)


# ============================================================
# 28. DID341 ZERO CHECK
# ============================================================

#
# CDC:
#
#     1-34 = 590
#     0    = 242 None
#     7777 = 2
#     9999 = 9
#
# Missing = 8,732
#

if int(
    df["DID341"]
    .between(
        1,
        34
    )
    .sum()
) != 590:

    raise ValueError(
        "DID341 1-34 range count differs."
    )


if int(
    df["DID341"].eq(0).sum()
) != 242:

    raise ValueError(
        "DID341 zero count differs."
    )


if int(
    df["DID341"].eq(7777).sum()
) != 2:

    raise ValueError(
        "DID341 refused count differs."
    )


if int(
    df["DID341"].eq(9999).sum()
) != 9:

    raise ValueError(
        "DID341 don't-know count differs."
    )


if int(
    df["DID341"].isna().sum()
) != 8732:

    raise ValueError(
        "DID341 missing count differs."
    )


print(
    "PASS: DID341 matches CDC."
)


# ============================================================
# 29. DID350 ZERO CHECK
# ============================================================

#
# CDC:
#
#     1-20 = 699
#     0    = 139 None
#     9999 = 5
#
# Missing = 8,732
#

if int(
    df["DID350"]
    .between(
        1,
        20
    )
    .sum()
) != 699:

    raise ValueError(
        "DID350 1-20 range count differs."
    )


if int(
    df["DID350"].eq(0).sum()
) != 139:

    raise ValueError(
        "DID350 zero count differs."
    )


if int(
    df["DID350"].eq(9999).sum()
) != 5:

    raise ValueError(
        "DID350 don't-know count differs."
    )


if int(
    df["DID350"].isna().sum()
) != 8732:

    raise ValueError(
        "DID350 missing count differs."
    )


print(
    "PASS: DID350 matches CDC."
)


# ============================================================
# 30. DID350 / DIQ350U ROUTING
# ============================================================

foot_positive = (
    df["DID350"]
    .between(
        1,
        20
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


foot_unit_present = (
    df["DIQ350U"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        foot_positive
        !=
        foot_unit_present
    )
) != 0:

    raise ValueError(
        "DIQ350U routing differs from DID350."
    )


check_frequency(
    "DIQ350U",
    {
        1: 487,
        2: 152,
        3: 46,
        4: 14
    },
    8876
)


# ============================================================
# 31. EYE EXAM / RETINOPATHY CHECK
# ============================================================

check_frequency(
    "DIQ360",
    {
        1: 82,
        2: 440,
        3: 129,
        4: 116,
        5: 64,
        9: 15
    },
    8729
)


check_frequency(
    "DIQ080",
    {
        1: 194,
        2: 644,
        9: 8
    },
    8729
)


# ============================================================
# 32. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original XPT contained zero ordinary numeric zeros.
#
# Therefore after correction the only zeros should be
# exactly the 582 validated restored positions.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 582:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 582 validated restorations."
)


# ============================================================
# 33. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY numerical modifications allowed:
#
#     validated tiny artifact -> 0
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 54 variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 34. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# 53 whole-number fields lose unnecessary ".0".
#
# DIQ280 stays continuous.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 35. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 36. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 54 DIQ_I variables."
)


# ============================================================
# 37. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 38. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 39. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 40. .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT globally check DIQ280 for ".0".
#
# DIQ280 is a genuine continuous A1C field.
#
# Only the 53 proven whole-number fields should
# contain no ".0".
#

print(
    "\n--- INTEGER FIELD .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' in integer/code fields:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' in "
    "any of the 53 whole-number fields."
)


# ============================================================
# 41. LEGITIMATE DIQ280 .0 VALUES
# ============================================================

#
# DIQ280 is a mixed continuous/special-code field.
#
# The attachment contains:
#
#     269 fractional observations
#     372 whole-looking observations
#
#
# Since DIQ280 remains float, whole-looking values
# legitimately remain with ".0".
#
# Examples:
#
#     5.0
#     6.0
#     7.0
#
# Special codes in this float column may also appear:
#
#     777.0
#     999.0
#
# We do NOT globally strip them because doing so would
# require altering formatting inside a genuine continuous
# decimal field.
#

diq280_dot_zero_count = int(
    (
        raw_csv_df["DIQ280"].ne("")
        &
        raw_csv_df["DIQ280"].str.endswith(".0")
    )
    .sum()
)


print(
    "\n--- LEGITIMATE DIQ280 .0 TOKENS ---"
)

print(
    "Count:",
    f"{diq280_dot_zero_count:,}"
)


if diq280_dot_zero_count != 372:

    raise ValueError(
        "Unexpected DIQ280 '.0' token count."
    )


print(
    "PASS: 372 legitimate whole-looking "
    "DIQ280 float values preserved."
)


# ============================================================
# 42. EXACT DIQ280 DECIMAL FIDELITY
# ============================================================

expected_a1c = (
    expected_df["DIQ280"]
    .to_numpy(
        dtype=float
    )
)


exported_a1c = (
    roundtrip_df["DIQ280"]
    .to_numpy(
        dtype=float
    )
)


a1c_exact = np.allclose(
    expected_a1c,
    exported_a1c,
    rtol=0,
    atol=0,
    equal_nan=True
)


if not a1c_exact:

    raise ValueError(
        "DIQ280 decimal fidelity failed."
    )


print(
    "\nPASS: Every DIQ280 numeric value "
    "preserved exactly."
)


# ============================================================
# 43. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "the 582 validated zero values."
)


# ============================================================
# 44. FINAL DIQ280 RANGE CHECK
# ============================================================

final_a1c = pd.to_numeric(
    raw_csv_df["DIQ280"]
    .where(
        raw_csv_df["DIQ280"] != "",
        np.nan
    ),
    errors="coerce"
)


if int(
    final_a1c
    .between(
        2,
        18.5
    )
    .sum()
) != 404:

    raise ValueError(
        "Final DIQ280 valid range count changed."
    )


if int(
    final_a1c.eq(777).sum()
) != 2:

    raise ValueError(
        "Final DIQ280 refused count changed."
    )


if int(
    final_a1c.eq(999).sum()
) != 235:

    raise ValueError(
        "Final DIQ280 don't-know count changed."
    )


print(
    "PASS: Final DIQ280 distribution preserved."
)


# ============================================================
# 45. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 46. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 47. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "DIQ_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Questionnaire - Diabetes"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Tiny artifact variables:",
    list(
        EXPECTED_TINY_COUNTS.keys()
    )
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "DID250 zeros:",
    int(
        df["DID250"].eq(0).sum()
    )
)


print(
    "DID260 zeros:",
    int(
        df["DID260"].eq(0).sum()
    )
)


print(
    "DID341 zeros:",
    int(
        df["DID341"].eq(0).sum()
    )
)


print(
    "DID350 zeros:",
    int(
        df["DID350"].eq(0).sum()
    )
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "DIQ280 nonmissing:",
    f"{a1c_nonmissing:,}"
)


print(
    "DIQ280 missing:",
    f"{a1c_missing:,}"
)


print(
    "DIQ280 valid A1C values:",
    a1c_valid_range
)


print(
    "DIQ280 fractional observations:",
    f"{a1c_fractional:,}"
)


print(
    "DIQ280 whole-looking observations:",
    f"{a1c_whole_looking:,}"
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "DIQ280 numeric differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0' in 53 integer fields:",
    "0 expected"
)


print(
    "DIQ280 '.0' values:",
    "ALLOWED / PRESERVED"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "DIQ280 remains a genuine continuous "
    "questionnaire A1C value."
)


print(
    "No genuine continuous decimal value was "
    "rounded or converted to integer."
)

NHANES 2015-2016 DIQ_I XPT -> CSV
Rows: 9,575
Columns: 54
PASS: Structure = 9,575 rows x 54 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: ['DIQ280']
Whole-number/code fields: 53

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 582
Variables containing tiny values: 4

Tiny values by variable:
DID341    242
DID260    188
DID350    139
DID250     13

PASS: Exact 582-value tiny-zero pattern across 4 variables validated.

--- VALIDATED ZERO RESTORATION ---
DID250: 13 restored to 0
DID260: 188 restored to 0
DID341: 242 restored to 0
DID350: 139 restored to 0

PASS: 582 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
DIQ010: 0 fractional observations
DID040: 0 fractional observations
DIQ160: 0 fractional observations
DIQ170: 0 fractional observations
DIQ172: 0 fractional observations
DIQ175A: 0 fractional observations
DIQ175